In [13]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

In [14]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [15]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip","r")
zip.extractall("/content")
zip.close()

In [16]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [20]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [22]:
age = []
gender = []
img_path = []
for file in os.listdir(folder_path):
    age.append(int(file.split('_')[0]))
    gender.append(int(file.split('_')[1]))
    img_path.append(file)

In [23]:
len(age)

23708

In [24]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [25]:
df.shape

(23708, 3)

In [26]:
df.head()

,age,gender,img
0,68,0,68_0_1_20170117172339634.jpg.chip.jpg
1,45,0,45_0_3_20170105163539339.jpg.chip.jpg
2,29,1,29_1_1_20170113000843651.jpg.chip.jpg
3,13,0,13_0_0_20170103200550455.jpg.chip.jpg
4,24,1,24_1_3_20170104231939707.jpg.chip.jpg


In [27]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [28]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [30]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [32]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [33]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [34]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [35]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [37]:
def fix_generator(generator):
    while True:
        x, y = next(generator)
        yield x, tuple(y)

In [38]:
history = model.fit(
    fix_generator(train_generator),
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=fix_generator(test_generator),
    validation_steps=len(test_generator)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 255s 385ms/step - age_loss: 15.3571 - age_mae: 15.3571 - gender_accuracy: 0.5151 - gender_loss: 0.8532 - loss: 99.8230 - val_age_loss: 15.7370 - val_age_mae: 15.7371 - val_gender_accuracy: 0.5235 - val_gender_loss: 0.6921 - val_loss: 84.2498
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 367ms/step - age_loss: 15.0044 - age_mae: 15.0044 - gender_accuracy: 0.5227 - gender_loss: 0.6972 - loss: 84.0265 - val_age_loss: 14.6873 - val_age_mae: 14.6873 - val_gender_accuracy: 0.5237 - val_gender_loss: 0.6923 - val_loss: 83.2286
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 366ms/step - age_loss: 14.9454 - age_mae: 14.9454 - gender_accuracy: 0.5223 - gender_loss: 0.6972 - loss: 83.9638 - val_age_loss: 15.4113 - val_age_mae: 15.4113 - val_gender_accuracy: 0.5213 - val_gender_loss: 0.6922 - val_loss: 83.9433
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 368ms/step - age_loss: 14.8506 - age_mae: 14.8506 - gender_accuracy: 0.5222 - gender_loss: 0.6923 - loss: 83.